In [1]:

import re
import json
import csv
from datetime import datetime, timezone

# SQL error indicators
SQL_ERROR_PATTERNS = [
    r"SQL syntax.*error",
    r"mysql_fetch",
    r"SQLite3?::",
    r"ORA-\d{5}",
    r"PostgreSQL.*ERROR",
    r"ODBC SQL Server Driver",
]

def detect_sql_errors(response_text):
    """Check text for common SQL error indicators."""
    findings = []

    for pattern in SQL_ERROR_PATTERNS:
        if re.search(pattern, response_text, re.IGNORECASE):
            findings.append(pattern)

    return findings


def scan_demo(samples):
    """Analyze sample responses without sending network requests."""
    results = []

    for sample in samples:
        findings = detect_sql_errors(sample["response"])

        results.append({
            "test_name": sample["name"],
            "possible_sql_error": bool(findings),
            "matched_indicators": findings,
            "checked_at": datetime.now(timezone.utc).isoformat()
        })

    return results


# Simulated responses — no website is contacted
demo_samples = [
    {
        "name": "Normal response",
        "response": "Welcome! Login successful."
    },
    {
        "name": "Simulated SQL error",
        "response": "SQL syntax error near unexpected token."
    },
    {
        "name": "Simulated database error",
        "response": "ORA-00933: SQL command not properly ended"
    },
    {
        "name": "Generic error",
        "response": "Something went wrong. Please try again."
    }
]

results = scan_demo(demo_samples)

print("SQL Injection Scanner — Offline Demo")
print("-" * 42)

for result in results:
    status = (
        "Possible SQL error indicator"
        if result["possible_sql_error"]
        else "No known indicator found"
    )

    print(f"\nTest: {result['test_name']}")
    print(f"Result: {status}")

with open("scan_report.json", "w", encoding="utf-8") as file:
    json.dump(results, file, indent=4)

with open("scan_report.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "test_name",
            "possible_sql_error",
            "matched_indicators",
            "checked_at"
        ]
    )
    writer.writeheader()

    for result in results:
        row = result.copy()
        row["matched_indicators"] = ", ".join(
            row["matched_indicators"]
        )
        writer.writerow(row)

print("\nReports saved: scan_report.json and scan_report.csv")


SQL Injection Scanner — Offline Demo
------------------------------------------

Test: Normal response
Result: No known indicator found

Test: Simulated SQL error
Result: Possible SQL error indicator

Test: Simulated database error
Result: Possible SQL error indicator

Test: Generic error
Result: No known indicator found

Reports saved: scan_report.json and scan_report.csv
